# 02 — Signal Analysis

Now we study the electrical characteristics of each appliance in detail.

**This notebook:**
- Computes steady-state power distributions per appliance
- Analyzes ON/OFF transition sharpness
- Studies periodic behavior (e.g. compressor cycling)
- Plots power histograms per appliance

**This is the EE part.** Understanding *why* loads look different is what lets you engineer meaningful features instead of just throwing raw data at a model.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import medfilt
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
df = pd.read_csv('../data/processed/house1_7days.csv', index_col=0, parse_dates=True)
print(df.shape)
df.head(3)

## Steady-state power distributions

For each appliance, when it is ON, what range of power does it draw?

Resistive loads (heaters, incandescent bulbs) are narrow and stable.
Motor loads (fridge, washer) are wider because the motor load varies.

In [ ]:
appliance_cols = [c for c in df.columns if c != 'mains']

fig, axes = plt.subplots(len(appliance_cols), 1, figsize=(10, 3*len(appliance_cols)))
if len(appliance_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, appliance_cols):
    # only plot when appliance is ON (power > 10W)
    on_data = df[col][df[col] > 10]
    if len(on_data) == 0:
        ax.set_title(f'{col} — no ON data', fontsize=10)
        continue
    ax.hist(on_data.values, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(on_data.mean(), color='crimson', linestyle='--', linewidth=1.5,
               label=f'mean={on_data.mean():.0f}W')
    ax.set_title(col, fontsize=10, fontweight='bold')
    ax.set_xlabel('Power (W)')
    ax.legend(fontsize=8)

plt.suptitle('Steady-State Power Distributions (when ON)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../outputs/02_power_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## ON duration analysis

How long does each appliance stay ON per activation?

Microwave: typically 1–5 minutes.
Fridge: 10–25 minutes per compressor cycle.
Washer: 45–90 minutes per wash cycle.

Duration is a strong discriminating feature.

In [ ]:
def get_on_durations(series, threshold=10, min_duration_s=5):
    '''
    Extract durations (in seconds) of each ON event for a given appliance.
    An ON event = consecutive samples above threshold.
    '''
    is_on = (series > threshold).astype(int)
    changes = is_on.diff().fillna(0)

    durations = []
    start = None

    for ts, val in changes.items():
        if val == 1:   # turned ON
            start = ts
        elif val == -1 and start is not None:  # turned OFF
            dur = (ts - start).seconds
            if dur >= min_duration_s:
                durations.append(dur)
            start = None

    return durations

# compute for each appliance
duration_data = {}
for col in appliance_cols:
    durs = get_on_durations(df[col])
    if len(durs) > 3:
        duration_data[col] = durs
        print(f'{col}: {len(durs)} events, median duration = {np.median(durs):.0f}s')

In [ ]:
if duration_data:
    fig, ax = plt.subplots(figsize=(10, 5))
    labels_list = list(duration_data.keys())
    data_list   = [np.clip(duration_data[k], 0, 3600) for k in labels_list]  # cap at 1hr

    bp = ax.boxplot(data_list, patch_artist=True, vert=True,
                    boxprops=dict(facecolor='lightsteelblue', color='steelblue'),
                    medianprops=dict(color='navy', linewidth=2))
    ax.set_xticklabels(labels_list, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('ON Duration (seconds)', fontsize=11)
    ax.set_title('ON Duration per Appliance (capped at 3600s)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../outputs/02_on_durations.png', dpi=150)
    plt.show()

## Compressor cycling — fridge behavior

The refrigerator compressor cycles on and off repeatedly throughout the day.
This creates a distinctive periodic pattern — very different from a light switch.

Plotting 6 hours of fridge data to see the cycle:

In [ ]:
# identify fridge column — adjust name if needed
fridge_candidates = [c for c in appliance_cols if 'fridge' in c.lower() or 'refrigerator' in c.lower()]

if fridge_candidates:
    fridge_col = fridge_candidates[0]
    fridge_6h = df[fridge_col].iloc[:6*3600]

    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(fridge_6h.index, fridge_6h.values, color='steelblue', linewidth=0.8)
    ax.set_title(f'{fridge_col} — 6 Hour View (Compressor Cycling)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Watts')
    ax.set_xlabel('Time')
    plt.tight_layout()
    plt.savefig('../outputs/02_fridge_cycling.png', dpi=150)
    plt.show()
else:
    print('No fridge column found — check your labels.dat for the exact name')

## Summary: Key signal characteristics per load type

| Load Type | Steady-State Power | Transition | Duration | Periodicity |
|---|---|---|---|---|
| Refrigerator | 100–200W | Gradual (motor) | 10–25 min | Yes (compressor) |
| Microwave | 1000–1500W | Instant | 1–5 min | No |
| Washer/Dryer | 400–5000W | Gradual | 45–90 min | No |
| Lighting | 60–300W | Instant | Variable | No |
| Dishwasher | 1200–1500W | Gradual | 60–90 min | No |

These differences become our features in notebook 04.